
## Structured Output

Structured output ka matlab hai model ke response ko ek fixed format ya structure mein lena.

Hum model ko ek specific schema follow karne ko kehte hain, taaki output ko easily read, parse aur aage code mein use kar sakein.

LangChain structured output ke liye different schema types aur methods provide karta hai.

### Pydantic

Pydantic ek Python library hai jo output ke data ka structure define aur validate karne mein help karti hai.

Iske main benefits:
- **Field validation:** Check karta hai ki data sahi type aur format mein hai ya nahi.
- **Descriptions:** Har field ke baare mein information define kar sakte hain.
- **Nested structures:** Ek structure ke andar doosra structure define kar sakte hain.


In [1]:
import os 
from langchain.chat_models import init_chat_model
#environment s  groq api key read krna
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
#groq chat model inutialize kro
model=init_chat_model("groq:openai/gpt-oss-120b")
#model call
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000268110E4830>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000268110E5550>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel,Field # pydantic is a base class for creating pydantic model
class Movie(BaseModel):#ye parent class h basemodel
    title:str=Field(description="the title of the movie")
    year:int=Field(description="the year in which it was released")#ye title year ye sb field h jiska description dere 
    #taki llm smjh paye
    director:str=Field(description="director of movie")
    rating:int=Field(description="rating movie got")

In [11]:
model_with_structure=model.with_structured_output(Movie, method="json_schema")
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000268110E4830>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000268110E5550>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'response_format': {'type': 'json_schema', 'json_schema': {'name': 'Movie', 'schema': {'title': 'Movie', 'description': '', 'properties': {'ti

In [12]:
model.invoke("give info about the movie")

AIMessage(content='Sure! Could you let me know which movie you’d like information about? (Title, year, or any details you have will help me give you the most relevant info.)', additional_kwargs={'reasoning_content': 'The user says "give info about the movie". They didn\'t specify which movie. Could be ambiguous. As assistant, we need to ask for clarification which movie they refer to. According to policy, we should ask clarifying question. Also note that we should not assume. So respond asking which movie they want info about.'}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 76, 'total_tokens': 185, 'completion_time': 0.227830213, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.002977234, 'prompt_tokens_details': None, 'queue_time': 0.308206886, 'total_time': 0.230807447}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f73454f048', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provi

In [ ]:
response=model_with_structure.invoke("give me info about movie 3 idiots")#ye ek schema means structure m ans dera h
#ar validate v kr rha, qki year m int hi ayga to validate v hora j
response

Movie(title='3 Idiots', year=2009, director='Rajkumar Hirani', rating=8)

# message output alongside parsed structure(mtlb raw form m jo model h wo jis trh s display krta h wo v dikhega )


In [ ]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="the title of the movie")
    year:int=Field(description="the year in which it was released")#ye title year ye sb field h jiska description dere 
        #taki llm smjh paye
    director:str=Field(description="director of movie")
    rating:int=Field(description="rating movie got")
model_with_structure=model.with_structured_output(Movie,method="json_schema",include_raw=True)
response=model_with_structure.invoke("give me info about 3 idiots movie")
response#ab phle ki trh wo ai msg v dikhega jo normal modle m dikhta h

{'raw': AIMessage(content='{"title":"3 Idiots","year":2009,"director":"Rajkumar Hirani","rating":9}', additional_kwargs={'reasoning_content': 'The user wants info about "3 idiots" movie. The system says we need to output JSON following the "Movie" schema. We need to include fields: title, year, director, rating. Provide info about the movie. Must output only final JSON object, compact formatting, no extra text. Ensure JSON is valid.\n\nWe need to fill with appropriate data. "3 Idiots" is a 2009 Indian Hindi-language comedy-drama film directed by Rajkumar Hirani. Rating? Could be maybe 8 (out of 10). Use integer rating. Provide rating maybe 9? Use integer. Let\'s choose rating 9.\n\nThus JSON: {"title":"3 Idiots","year":2009,"director":"Rajkumar Hirani","rating":9}\n\nMake sure compact, no spaces? It can have spaces but it\'s okay. Provide exactly that.'}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 219, 'total_tokens': 432, 'completion_time': 0.4511018

## nested structure

In [28]:
from pydantic import BaseModel,Field
class Actor(BaseModel):
    name:str
    role:str
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float|None = Field(None,description="budget in millions usd")

model_with_structure=model.with_structured_output(MovieDetails,method="json_schema")
response=model_with_structure.invoke("give info about movie")
response


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

# typedict(another puthon library)

## TypedDict

`TypedDict` Python ke built-in `typing` module ka feature hai, jo dictionary ke expected structure aur har key ke data type ko define karta hai.

Ye tab useful hai jab humein structured output chahiye, lekin **runtime par data validation** ki zaroorat nahi hoti.

### Simple example

- `title`: movie ka naam (`str`)
- `year`: release year (`int`)
- `rating`: movie ki rating (`float`)

**Pydantic se difference:** Pydantic runtime par data validate kar sakta hai, jabki `TypedDict` mainly expected structure aur types ko describe karta hai.


In [22]:
from typing_extensions import TypedDict,Annotated #annotated k mtlb hota h extra info hona
class MovieDict(TypedDict):
    """A movie with details."""
    title:Annotated[str,...,"the title of movie"]
    year:Annotated[int,...,"year in which movie released"]
    director:Annotated[str,...,"director of movie"]#... k mtlv extra are v kch ho skta h
    rating:Annotated[int,...,"rating of movie"]
movie_with_typedict=model.with_structured_output(MovieDict,method="json_schema")
response=movie_with_typedict.invoke("give info about movie")
response

{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 9}

In [29]:
from typing_extensions import TypedDict
class Actor(TypedDict):
    name:str
    role:str
class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float|None = Field(None,description="budget in millions usd")

model_with_structure=model.with_structured_output(MovieDetails,method="json_schema")
response=model_with_structure.invoke("give info about movie")
response


{'title': 'Inception',
 'year': 2010,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'}],
 'genres': ['Action', 'Sci-Fi', 'Thriller'],
 'budget': 160000000}

In [ ]:
pydantic check krta h agr match ni hua int string to ya to convert if possible ya to validation error, ar typedict chek ni krta, int mention hok string v rheg ato string display kr dega

structure model k profile nid ekh skte but normal model k dekh skte

In [30]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

# dataclasses

DataClass Python mein ek tarika hai jisse hum data store karne wali class easily bana sakte hain.

Iske liye @dataclass decorator use hota hai.
normally class m init s data iniialize hota h pr yaha ese hi ho jata automatically

In [33]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


hmlg integrate krenge create agent with this format of class pydantic

In [36]:

from pydantic import BaseModel, Field
from langchain.agents import create_agent

# Contact information ka structure define karna
class ContactInfo(BaseModel):
    """Contact info about a person"""
    name: str = Field(description="Name of a person")
    email: str = Field(description="Email of a person")
    phone: str = Field(description="Phone number of a person")

# Agent create karna aur batana ki output ContactInfo format mein chahiye
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

# User ka message agent ko dena
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, 8521060876"
        }
    ]
})
result




{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, 8521060876', additional_kwargs={}, response_metadata={}, id='87dee63f-87f0-441a-806c-eaee7c26404b'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"8521060876"}', additional_kwargs={'reasoning_content': 'The user asks: "Extract contact info from: John Doe, john@example.com, 8521060876". We need to output JSON matching the ContactInfo schema. Must include fields name, email, phone. Use compact JSON formatting. So output: {"name":"John Doe","email":"john@example.com","phone":"8521060876"} Ensure all required fields present. That\'s it.'}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 231, 'total_tokens': 343, 'completion_time': 0.23340266, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.01121543, 'prompt_tokens_details': None, 'queue_time': 0.212670881, 'total_time': 0.24461809}, 'model_name': 'openai/gpt-oss-120b'

In [37]:
# Structured response print karna
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='8521060876'


with the help of dataclass too, u can do the same thing

In [39]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact info of a person"""
    name: str  # Name of a person
    email: str  # Email of a person
    phone: str  # Phone number of a person


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

Middleware kya hota hai?
Middleware ek aisa layer hai jo AI Agent ke kaam karne ke process ko control aur monitor karta hai.

Jaise ek security guard check karta hai ki kaun andar ja raha hai, waise hi middleware agent ke process par nazar rakhta hai aur zaroorat ke hisaab se changes karta hai.
Middleware ke uses
1. Tracking agent behavior

Agent kya kar raha hai, kitne tokens use kar raha hai aur error kahan aa raha hai, ye track karna. Ismein logging, analytics aur debugging aate hain.

2. Transforming prompts, tool selection and output formatting
- Prompt ko modify karna.
- Agent ko sahi tool choose karne mein help karna.
- Output ko required format mein badalna.
3. Retries, fallbacks and early termination
- Retries: Error aane par dobara try karna.
- Fallbacks: Ek model fail ho toh doosra model use karna.
- Early termination: Zaroorat padne par agent ka process pehle hi rok dena.
4. Rate limits, guardrails and PII detection
- Rate limits: Requests ki sankhya limit karna.
- Guardrails: Agent ko rules aur safety boundaries ke andar rakhna.
- PII detection: Personal information, jaise phone number aur email, ko pehchanna.
Ek line mein yaad karo
Middleware = AI Agent ke behavior ko control, monitor aur secure karne wala layer.


# Middleware in LangChain

## 1. Normal Agent Flow

```mermaid
flowchart TD
    A[User Request] --> B[Model]
    B --> C[Tools]
    C --> D[Observation]
    D --> B
    B --> E[Final Response]

    style A fill:#dbeafe,stroke:#2563eb,color:#111
    style B fill:#dbeafe,stroke:#2563eb,color:#111
    style C fill:#dcfce7,stroke:#16a34a,color:#111
    style D fill:#fef3c7,stroke:#d97706,color:#111
    style E fill:#dcfce7,stroke:#16a34a,color:#111
```

### Explanation
1. **Request:** User agent ko input deta hai.
2. **Model:** Input samajhkar decide karta hai ki kya karna hai.
3. **Tools:** Model ke kehne par tool koi action perform karta hai.
4. **Observation:** Tool ka result model ko milta hai.
5. **Final Response:** Model user ko final answer deta hai.

> Model zaroorat ke hisaab se tool ko dobara call kar sakta hai.

---

## 2. Agent with Middleware

```mermaid
flowchart TD
    A[Request] --> B[before_agent]
    B --> C[before_model]
    C --> D[wrap_model_call]
    D --> E[Model]
    E --> F[after_model]
    F --> G{Tool call needed?}
    G -- Yes --> H[wrap_tool_call]
    H --> I[Tools]
    I --> C
    G -- No --> J[after_agent]
    J --> K[Final Result]

    style A fill:#dbeafe,stroke:#2563eb,color:#111
    style B fill:#ffedd5,stroke:#ea580c,color:#111
    style C fill:#ffedd5,stroke:#ea580c,color:#111
    style D fill:#dbeafe,stroke:#2563eb,color:#111
    style E fill:#dbeafe,stroke:#2563eb,color:#111
    style F fill:#ffedd5,stroke:#ea580c,color:#111
    style H fill:#dcfce7,stroke:#16a34a,color:#111
    style I fill:#dcfce7,stroke:#16a34a,color:#111
    style J fill:#ffedd5,stroke:#ea580c,color:#111
    style K fill:#dcfce7,stroke:#16a34a,color:#111
```

### Explanation in Hindi

Middleware agent ke process ke beech mein checkpoints provide karta hai. In checkpoints ko **hooks** kehte hain.

- **before_agent:** Agent ka kaam shuru hone se pehle execute hota hai.
- **before_model:** Model ko call karne se pehle execute hota hai.
- **wrap_model_call:** Model call ko control ya modify kar sakta hai.
- **after_model:** Model ka response aane ke baad execute hota hai.
- **wrap_tool_call:** Tool call ko control ya modify kar sakta hai.
- **after_agent:** Agent ka kaam complete hone ke baad execute hota hai.

### Example

Agar user personal information poochta hai, middleware us information ko check kar sakta hai. Agar request limit cross ho jaaye, toh middleware request ko rok bhi sakta hai.

### Main Difference

| Normal Agent | Agent with Middleware |
|---|---|
| Basic request-model-tool flow | Extra checkpoints available |
| Limited control over the flow | Flow ko inspect aur modify kar sakte hain |
| Tool aur model calls hote hain | Calls par logging, limits aur guardrails laga sakte hain |


Built-in Middleware in LangChain
Built-in middleware are predefined middleware provided by LangChain to add extra functionality to an AI agent without implementing everything from scratch.
1. Summarization Middleware
Jab conversation mein bahut saare messages ho jaate hain, toh model ko poori conversation bhejna costly ho sakta hai aur context limit exceed ho sakti hai.

Summarization middleware purane messages ka summary banata hai, jisse important information retain rahe aur messages ka size kam ho.

Example: User ne agent se 50 questions poochhe. Middleware purani conversation ko summarize karke important details rakhta hai, aur agent aage kaam continue karta hai.
2. Human-in-the-Loop Middleware
Ye middleware agent ke kisi action ko execute karne se pehle human se approval ya feedback lene ki facility deta hai.

Example: Agent ko email send karna hai. Email send karne se pehle middleware user se approval maang sakta hai.

Iska use sensitive actions, jaise payment, email sending ya data deletion, mein kiya ja sakta hai.
3. Model Call Limit Middleware
Ye middleware agent ke model calls ki maximum limit set karta hai.

Example: Agar maximum model calls ki limit 10 hai, toh agent ek run mein limit cross karke endlessly model ko call nahi kar sakta.

Isse unnecessary API calls, token usage aur cost ko control karne mein help milti hai.

# summarization middleware

In [54]:
from langchain.agents import create_agent
#- SummarizationMiddleware ek built-in middleware hai.
#- Ye conversation bahut lambi hone par messages ko summarize karne mein help karta hai.
#Kyun? Taaki har purana message detail mein carry karne ki zaroorat na pade aur important context bana rahe.
from langchain.agents.middleware import SummarizationMiddleware
#- InMemorySaver conversation ki state ko memory mein save karne ke liye hai.
#- Isse agent ki state ko same conversation mein aage continue kiya ja sakta hai.
#Kyun? Taaki agent ke paas pichhli conversation ka context reh sake.

#Note: Iski memory Python process band hone par generally persist nahi karti.
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

###messagebased summarization
agent=create_agent(model="groq:openai/gpt-oss-120b",
                   checkpointer=InMemorySaver(),#save krta h convo ko my nam eis ritika, bd m what is ur name k ans hoga
                   middleware=[SummarizationMiddleware(
                       model="groq:openai/gpt-oss-120b",
                       trigger=("messages",10),#jb 10 msg ayga tb triger hoga mtlb summarize hoga
                       keep=("messages",4)#4 msg original form m rhe ar 6 k summary hoga
                   )]
                   )


In [47]:
#run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [50]:
# Questions ki list
questions = [
    "What is 2 * 2?",
    "What is 4 + 3?",
    "What is 8 * 8?",
    "What is 5 + 8?",
    "What is 7 * 8?",
    "What is 8 / 8?",  # Ye question alag hai
    "What is 7 * 7?"
]

# Same conversation ko continue karne ke liye
config = {
    "configurable": {
        "thread_id": "1"
    }
}

# Har question ko ek-ek karke agent ko bhejna
for q in questions:

    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config=config
    )

 # Pura response print karna
print(f"Messages: {response}")

# Messages ki total sankhya print karna
print(f"Number of messages: {len(response['messages'])}")
          

Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nProvide correct answers to the user's arithmetic questions.\n\n## SUMMARY\n- Answered: “What is 4\u202f+\u202f3?” → 7.  \n- Answered: “What is 8\u202f×\u202f8?” → 64.  \n- Answered: “What is 5\u202f+\u202f8?” → 13.  \n- Pending: “What is 7\u202f×\u202f8?” (needs correct answer 56).  \n- Earlier, an incorrect answer for “7\u202f×\u202f8” was given as 56.56; the correct value is 56.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\n- Provide the correct answer to “What is 7\u202f×\u202f8?” → 56, and optionally acknowledge the prior mistake.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='49ee72af-a6b6-4e68-9178-6a931e5c987d'), AIMessage(content='7\u202f×\u202f8\u202f=\u202f56.', additional_kwargs={'reasoning_content': 'User asks "What is 7 * 8?" Previously answered incorrectly as 56.56. Need to give correct answer: 56. Also maybe acknowledge correction? J

based on token

In [55]:
# Required imports
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver


# Tool: City ke hotels ki details return karega
@tool
def search_hotels(city: str) -> str:
    """Search hotels and return hotel details."""

    # Long response diya hai taaki token usage badhe
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


# Agent create karna
agent = create_agent(
    model="groq:openai/gpt-oss-120b",  # Main AI model

    tools=[search_hotels],  # Agent ko hotel search tool dena

    checkpointer=InMemorySaver(),  # Conversation state memory mein save karna

    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",  # Summary banane wala model

            # Token count 550 ke aas-paas hone par summarization trigger
            trigger=("tokens", 550),

            # Summary ke baad recent context ke lagbhag 200 tokens preserve
            keep=("tokens", 200)
        )
    ]
)


# Same conversation ko continue karne ke liye thread ID
config = {
    "configurable": {
        "thread_id": "test-1"
    }
}


# Approximate token counter
def count_tokens(messages):
    # Sabhi messages ke content ki total character length
    total_chars = sum(len(str(m.content)) for m in messages)

    # Approximation: 4 characters ko lagbhag 1 token maana hai
    return total_chars // 4




In [56]:
# Run test: alag-alag cities ke hotels search karna
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

# Har city ke liye loop chalega
for city in cities:

    # City ka naam user query mein daal kar agent ko bhejna
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"Find hotels in {city}")
            ]
        },
        config=config  # Same conversation ko continue karna
    )

    # Response ke messages ka approximate token count nikalna
    tokens = count_tokens(response["messages"])

    # City, approximate tokens aur messages ki count print karna
    print(
        f"{city}: ~{tokens} tokens, "
        f"{len(response['messages'])} messages"
    )

    # Agent ke messages aur summary ka content dekhna
    print(response["messages"])
    print("-" * 60)

Paris: ~280 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='f683c0c7-4216-4437-b0b7-c87e0b110946'), AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants hotels in Paris. Use function search_hotels.', 'tool_calls': [{'id': 'fc_6bd2f43e-253d-48d1-b5c2-e53e83e5593c', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 125, 'total_tokens': 166, 'completion_time': 0.086720801, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.027775103, 'prompt_tokens_details': None, 'queue_time': 1.84290409, 'total_time': 0.114495904}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8655ddce88', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17e00-422b-743a-a389-7aa9fa07c2cd-0', tool_calls=[{'nam

550 token k aaspas ya utna hi rhega to summary aa jayga

based on frcation now

In [58]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


# Hotel search tool
@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"""Hotels in {city}:
    Grand Hotel $350, City Inn $180, Budget Stay $75"""


# Agent with summarization middleware
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("fraction", 0.005),  # 0.5% context window
            keep=("fraction", 0.002),     # 0.2% context window
        ),
    ],
)

# Same conversation continue karne ke liye
config = {
    "configurable": {
        "thread_id": "test-1"
    }
}

In [59]:
def count_tokens(messages):
    # Har message ke content ki character length count karna
    return sum(len(str(m.content)) for m in messages) // 4

In [60]:
# Test cities
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:

    # Agent ko hotel search request bhejna
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )

    # Approximate tokens count karna
    tokens = count_tokens(response["messages"])

    # 128000 ko example context-window denominator maana hai
    fraction = tokens / 128000

    # City, token count aur messages ki sankhya print karna
    print(
        f"{city}: ~{tokens} tokens ({fraction:.4%}), "
        f"{len(response['messages'])} messages"
    )

    # Messages aur summary dekhna
    print(response["messages"])
    print("-" * 60)

Paris: ~230 tokens (0.1797%), 4 messages
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='7fa6d881-f211-4856-be75-7022f3dc9e83'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants hotels in Paris. We should use the search_hotels function with city "Paris".', 'tool_calls': [{'id': 'fc_322b8817-daa5-4029-8421-f36442815f08', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 120, 'total_tokens': 169, 'completion_time': 0.104000481, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.00576054, 'prompt_tokens_details': None, 'queue_time': 0.308996351, 'total_time': 0.109761021}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_96d96a151c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17e1f-80d7-71a

# human in the loop middleware(sumaarization k bd k topic middleware m)

# Human-in-the-Loop Middleware

## What is Human-in-the-Loop?

Human-in-the-Loop (HITL) is a middleware technique that pauses an AI agent's execution so a human can review, approve, edit, or reject an action before it is executed.

### Why do we need it?

- **Human Approval:** Important actions, such as financial transactions or database updates, can require approval before execution.
- **Compliance:** Some workflows require human supervision.
- **Human Feedback:** Humans can guide the agent during long-running conversations.

## How does it work?

1. The user gives a request to the AI agent.
2. The agent decides whether a tool needs to be called.
3. The middleware pauses execution before the tool runs.
4. A human reviews the proposed tool call.
5. The human approves, edits, or rejects the action.
6. The agent continues according to the decision.

## Example

Suppose a user asks an agent to transfer ₹5,000.

- The agent proposes a transfer using a payment tool.
- Human-in-the-loop middleware pauses the process.
- The human reviews the amount and recipient.
- If approved, the tool executes the transfer.
- If rejected, the transfer does not execute.

## Key Benefits

- Prevents unauthorized actions.
- Gives humans control over important decisions.
- Improves safety and reliability.
- Supports human supervision in sensitive workflows.

In [8]:

# create_agent: LangChain ka function hai jo AI agent banata hai.
from langchain.agents import create_agent

# HumanInTheLoopMiddleware: Jab kisi tool ko human approval chahiye,
# tab agent ko rok kar human se decision lene mein help karta hai.
from langchain.agents.middleware import HumanInTheLoopMiddleware

# InMemorySaver: Agent ki state ko memory mein save karta hai,
# taaki interruption ke baad agent ko resume kiya ja sake.
from langgraph.checkpoint.memory import InMemorySaver


# Tool 1: Email read karne ke liye function.
def read_email_tool(email_id: str) -> str:
    # email_id: Email ki ID input ke roop mein lega.
    # -> str: Batata hai ki function string return karega.
    # Ye docstring function ka purpose batati hai.
    """Mock function to read an email by its id"""

    # Abhi ye actual email nahi padhta, sirf sample message return karta hai.
    return f"Email content for ID: {email_id}"


# Tool 2: Email send karne ke liye function.
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    # recipient: Jise email bhejna hai, uska email address.
    # subject: Email ka subject.
    # body: Email ka main content.
    # -> str: Function string return karega.
    """Mock function to send an email"""

    # Abhi actual email send nahi hota.
    # Ye sirf confirmation jaisa sample message return karta hai.
    return f"Email sent to {recipient} with subject '{subject}'"


# create_agent() se AI agent create kar rahe hain.
agent = create_agent(

    # AI model jo user ki request samjhega aur decide karega
    # ki kaun-sa tool use karna hai.
    model="groq:openai/gpt-oss-120b",

    # Agent ko dono tools available karwa rahe hain.
    # AI zarurat ke hisaab se inmein se tool choose kar sakta hai.
    tools=[read_email_tool, send_email_tool],

    # Agent ki state save karne ke liye in-memory checkpointer.
    # Ye interruption ke baad conversation ko resume karne mein help karta hai.
    # Program restart hone par ye memory normally clear ho jaati hai.
    checkpointer=InMemorySaver(),

    # Middleware agent ke tool execution par additional control lagata hai.
    middleware=[
        HumanInTheLoopMiddleware(

            # interrupt_on decide karta hai ki kis tool par
            # human approval lena hai aur kis par nahi.
            interrupt_on={

                # Email send karne se pehle agent interrupt hoga.
                "send_email_tool": {

                    # Human ko teen decisions mein se choose karne ka option hai.
                    "allowed_decisions": [

                        # approve: Email send karne ki permission dena.
                        "approve",

                        # edit: Tool ke arguments, jaise recipient, subject ya body,
                        # ko modify karke action continue karna.
                        "edit",

                        # reject: Email send karne ki request reject karna.
                        "reject"
                    ]
                },

                # Email read karne ke liye human approval nahi chahiye.
                # False ka matlab hai ki is tool par HITL interrupt nahi hoga.
                "read_email_tool": False,

            }  # interrupt_on dictionary yahan end hoti hai.

        )  # HumanInTheLoopMiddleware yahan end hota hai.

    ]  # Middleware list yahan end hoti hai.

)  # create_agent() yahan end hota hai.


In [9]:

# HumanMessage user ke message ko represent karta hai.
from langchain_core.messages import HumanMessage

# Config agent ki conversation ko identify karne ke liye hai.
config = {
    "configurable": {
        # thread_id ek unique conversation ID hai.
        # Same ID se usi conversation ki saved state resume kar sakte hain.
        "thread_id": "test-approve"
    }
}

# Step 1: Request
# agent.invoke() agent ko input deta hai aur usse execute karta hai.
result = agent.invoke(
    {
        # messages mein user ka message pass kiya ja raha hai.
        "messages": [
            HumanMessage(
                # User agent ko email send karne ka instruction de raha hai.
                content=(
                    "Send email to john@test.com "
                    "with subject 'Hello' and body 'How are you?'"
                )
            )
        ]
    },

    # Config agent ko conversation ki ID deta hai.
    config=config
)

# Agent se mila poora result print hoga.
# Agar send_email_tool call hota hai, toh HITL interruption ki details
# result mein aa sakti hain.
print(result)


{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='b98bd21c-21c5-448d-b1d1-8e8b5dc2f126'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_65c641c2-6537-402f-944d-56d19a08d34f', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 174, 'total_tokens': 232, 'completion_time': 0.121648231, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.007225952, 'prompt_tokens_details': None, 'queue_time': 0.209750225, 'total_time': 0.128874183}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c3da0d4bb9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': '

In [ ]:

# Step 2: Approve
from langgraph.types import Command

# Check kar rahe hain ki agent ka result interrupt hua hai ya nahi.
# "__interrupt__" present hone ka matlab hai ki agent kisi
# human decision ka wait kar raha hai.
if "__interrupt__" in result:

    # User ko batayenge ki agent pause tha aur ab approve kar rahe hain.
    print("⏸️ Paused! Approving...")

    # Command(resume=...) se interrupted agent ko resume karenge.
    result = agent.invoke(
        Command(

            # Resume ke time human ka decision pass karte hain.
            resume={

                # "decisions" ek list hai jisme human ke decisions hote hain.
                "decisions": [

                    # "approve" ka matlab hai tool execute karne ki permission.
                    {"type": "approve"}#yaha reject likhte to reject ho jata
                ]
            }
        ),

        # Wahi config aur thread_id use karenge,
        # taaki agent ki pending conversation continue ho.
        config=config
    )

# Agent ke latest message ka content print karenge.
print(f"✅ Result: {result['messages'][-1].content}")


⏸️ Paused! Approving...
✅ Result: The email has been sent successfully. Let me know if there’s anything else you’d like to do!


reject k liye ab, test reject use krenge  

In [18]:

# create_agent: LangChain ka function hai jo AI agent banata hai.
from langchain.agents import create_agent

# HumanInTheLoopMiddleware: Jab kisi tool ko human approval chahiye,
# tab agent ko rok kar human se decision lene mein help karta hai.
from langchain.agents.middleware import HumanInTheLoopMiddleware

# InMemorySaver: Agent ki state ko memory mein save karta hai,
# taaki interruption ke baad agent ko resume kiya ja sake.
from langgraph.checkpoint.memory import InMemorySaver


# Tool 1: Email read karne ke liye function.
def read_email_tool(email_id: str) -> str:
    # email_id: Email ki ID input ke roop mein lega.
    # -> str: Batata hai ki function string return karega.
    # Ye docstring function ka purpose batati hai.
    """Mock function to read an email by its id"""

    # Abhi ye actual email nahi padhta, sirf sample message return karta hai.
    return f"Email content for ID: {email_id}"


# Tool 2: Email send karne ke liye function.
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    # recipient: Jise email bhejna hai, uska email address.
    # subject: Email ka subject.
    # body: Email ka main content.
    # -> str: Function string return karega.
    """Mock function to send an email"""

    # Abhi actual email send nahi hota.
    # Ye sirf confirmation jaisa sample message return karta hai.
    return f"Email sent to {recipient} with subject '{subject}'"


# create_agent() se AI agent create kar rahe hain.
agent = create_agent(

    # AI model jo user ki request samjhega aur decide karega
    # ki kaun-sa tool use karna hai.
    model="groq:openai/gpt-oss-120b",

    # Agent ko dono tools available karwa rahe hain.
    # AI zarurat ke hisaab se inmein se tool choose kar sakta hai.
    tools=[read_email_tool, send_email_tool],

    # Agent ki state save karne ke liye in-memory checkpointer.
    # Ye interruption ke baad conversation ko resume karne mein help karta hai.
    # Program restart hone par ye memory normally clear ho jaati hai.
    checkpointer=InMemorySaver(),

    # Middleware agent ke tool execution par additional control lagata hai.
    middleware=[
        HumanInTheLoopMiddleware(

            # interrupt_on decide karta hai ki kis tool par
            # human approval lena hai aur kis par nahi.
            interrupt_on={

                # Email send karne se pehle agent interrupt hoga.
                "send_email_tool": {

                    # Human ko teen decisions mein se choose karne ka option hai.
                    "allowed_decisions": [

                        # approve: Email send karne ki permission dena.
                        "approve",

                        # edit: Tool ke arguments, jaise recipient, subject ya body,
                        # ko modify karke action continue karna.
                        "edit",

                        # reject: Email send karne ki request reject karna.
                        "reject"
                    ]
                },

                # Email read karne ke liye human approval nahi chahiye.
                # False ka matlab hai ki is tool par HITL interrupt nahi hoga.
                "read_email_tool": False,

            }  # interrupt_on dictionary yahan end hoti hai.

        )  # HumanInTheLoopMiddleware yahan end hota hai.

    ]  # Middleware list yahan end hoti hai.

)  # create_agent() yahan end hota hai.


In [19]:

# HumanMessage user ke message ko represent karta hai.
from langchain_core.messages import HumanMessage

# Config agent ki conversation ko identify karne ke liye hai.
config = {
    "configurable": {
        # thread_id ek unique conversation ID hai.
        # Same ID se usi conversation ki saved state resume kar sakte hain.
        "thread_id": "test-reject"##reject kiye yaha
    }
}

# Step 1: Request
# agent.invoke() agent ko input deta hai aur usse execute karta hai.
result = agent.invoke(
    {
        # messages mein user ka message pass kiya ja raha hai.
        "messages": [
            HumanMessage(
                # User agent ko email send karne ka instruction de raha hai.
                content=(
                    "Send email to john@test.com "
                    "with subject 'Hello' and body 'How are you?'"
                )
            )
        ]
    },

    # Config agent ko conversation ki ID deta hai.
    config=config
)

# Agent se mila poora result print hoga.
# Agar send_email_tool call hota hai, toh HITL interruption ki details
# result mein aa sakti hain.
print(result)


{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a9494a8a-a53f-40e4-afa2-71f7591186ae'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using function. Use send_email_tool.', 'tool_calls': [{'id': 'fc_ec46aefd-2791-423f-8592-b191afa8b7ab', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.126386472, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.041906976, 'prompt_tokens_details': None, 'queue_time': 0.325476473, 'total_time': 0.168293448}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'mod

In [21]:

# Step 2: Approve
from langgraph.types import Command

# Check kar rahe hain ki agent ka result interrupt hua hai ya nahi.
# "__interrupt__" present hone ka matlab hai ki agent kisi
# human decision ka wait kar raha hai.
if "__interrupt__" in result:

    # User ko batayenge ki agent pause tha aur ab approve kar rahe hain.
    print("⏸️ Paused! Approving...")

    # Command(resume=...) se interrupted agent ko resume karenge.
    result = agent.invoke(
        Command(

            # Resume ke time human ka decision pass karte hain.
            resume={

                # "decisions" ek list hai jisme human ke decisions hote hain.
                "decisions": [

                    #  "reject" ka matlab hai tool execute karne ki request reject karna
                    {"type": "reject"}#yaha reject likhte to reject ho jata
                ]
            }
        ),

        # Wahi config aur thread_id use karenge,
        # taaki agent ki pending conversation continue ho.
        config=config
    )

# Agent ke latest message ka content print karenge.
print(f"✅ Result: {result['messages'][-1].content}")


✅ Result: I’m ready to send that email for you. Would you like me to go ahead and send it now?


In [22]:
# Saare messages aur unke types print karo.
for msg in result["messages"]:
    print("Type:", type(msg).__name__)
    print("Content:", msg.content)
    print("Tool calls:", getattr(msg, "tool_calls", []))
    print("-" * 40)

Type: HumanMessage
Content: Send email to john@test.com with subject 'Hello' and body 'How are you?'
Tool calls: []
----------------------------------------
Type: AIMessage
Content: 
Tool calls: [{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'fc_ec46aefd-2791-423f-8592-b191afa8b7ab', 'type': 'tool_call'}]
----------------------------------------
Type: ToolMessage
Content: User rejected the tool call for `send_email_tool` with id fc_ec46aefd-2791-423f-8592-b191afa8b7ab. The tool was not executed. Do not retry this tool call unless the user explicitly requests it.
Tool calls: []
----------------------------------------
Type: AIMessage
Content: I’m ready to send that email for you. Would you like me to go ahead and send it now?
Tool calls: []
----------------------------------------


edit k liye ab

In [23]:
#step 1-approve
# HumanMessage user ke message ko represent karta hai.
from langchain_core.messages import HumanMessage

# Conversation ki unique ID set karte hain.
config = {
    "configurable": {
        # Edit test ke liye alag thread_id.
        "thread_id": "test-edit"
    }
}

# Step 1: Request
# Agent ko email send karne ki request dete hain.
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Send email to john@test.com "
                    "with subject 'Hello' and body 'How are you?'"
                )
            )
        ]
    },
    config=config
)

# Agent ka result print karte hain.
print(result)

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c6ea2b56-a586-41bb-b7c0-4abea06bd7f1'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Use function.', 'tool_calls': [{'id': 'fc_ec903dd7-d90f-44a9-968b-f85f55d7a0ea', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.126120618, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.007008205, 'prompt_tokens_details': None, 'queue_time': 0.352589595, 'total_time': 0.133128823}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_02b0d31eca', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'mod

In [24]:
# Step 2: Edit
from langgraph.types import Command

# Check karte hain ki agent approval ke liye interrupt hua hai ya nahi.
if "__interrupt__" in result:

    # Batate hain ki ab email details edit karenge.
    print("⏸️ Paused! Editing...")

    # Agent ko resume karke email ke arguments edit karte hain.
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        # Edit decision select karte hain.
                        "type": "edit",

                        # Tool call ke arguments update karte hain.
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "riya@test.com",
                                "subject": "Updated Hello",
                                "body": "I hope you are doing well!"
                            }
                        }
                    }
                ]
            }
        ),
        # Same conversation ko resume karte hain.
        config=config
    )

# Final result ka last message print karte hain.
print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✅ Result: I’ve sent the email.  

**Recipient:** riya@test.com  
**Subject:** Updated Hello  
**Body:** I hope you are doing well!  

*Note: The system substituted the original request with a different recipient, subject, and body before the email was actually sent.*


In [25]:
# tm ar v middleware explore krskti h with syuntax in langchain.docs